
# Day 50 – State Space Models (SSM), S4 and Mamba

## Topics Covered
- Limitation of Transformers
- State Space Equations
- State Vectors
- Continuous and Discrete SSM
- Step Size (Δ) and impact on A, B, C
- Convolution View of SSM
- Linear State Space Layer
- HiPPO
- S4 (Structured State Space for Sequences)
- Limitations of SSM/S4
- Selective Copying & Induction Heads
- Mamba Architecture
- Selective Scan Algorithm
- Kernel Fusion & Hardware-Aware Design
- Jamba, Zamba and Hybrid Future

This notebook is theory-heavy and includes hands-on demonstrations using pretrained Hugging Face models.



# 1. Why Transformers Need Alternatives

Transformer attention complexity:

\[O(N^2)\]

Problems:

1. Quadratic memory growth
2. Expensive long-context inference
3. KV cache requirements
4. Poor streaming efficiency

Goal:

Build models with:

- Linear complexity
- Constant memory growth
- Long-range memory
- Fast inference

This leads to State Space Models (SSMs).



# 2. State Space Equations

Continuous-time system:

\[
\frac{dh(t)}{dt}=Ah(t)+Bx(t)
\]

Output:

\[
y(t)=Ch(t)
\]

Where:

- h(t) = state vector
- x(t) = input
- y(t) = output
- A = transition matrix
- B = input matrix
- C = output matrix



# 3. State Vectors

A state vector acts as compressed memory.

Transformer:
stores many previous tokens explicitly.

SSM:
stores history inside a compact state vector.

Example:

"The cat sat on the ..."

The state remembers grammar, semantics and context without storing every token.



# 4. Discrete State Space Model

Computers operate in discrete steps.

\[
h_t=\bar Ah_{t-1}+\bar Bx_t
\]

\[
y_t=Ch_t
\]

where:

\[
\bar A=e^{A\Delta}
\]

Step size Δ controls discretization quality.



# 5. Step Size Impact

Small Δ:
- More accurate
- Higher computation

Large Δ:
- Faster
- Less accurate

Discretization:

\[
\bar A=e^{A\Delta}
\]

\[
\bar B=A^{-1}(e^{A\Delta}-I)B
\]

Changing Δ changes memory retention and stability.


In [1]:

import numpy as np

A = np.array([[-0.5]])

for delta in [0.1, 1.0, 2.0]:
    A_bar = np.exp(A * delta)
    print(f"Delta={delta} -> A_bar={A_bar[0][0]:.4f}")


Delta=0.1 -> A_bar=0.9512
Delta=1.0 -> A_bar=0.6065
Delta=2.0 -> A_bar=0.3679



# 6. Convolution View

Expanding the recurrence shows:

y = K * x

where K is a learned kernel.

This allows efficient sequence processing using convolution and FFT-based acceleration.



# 7. Linear State Space Layer

Pipeline:

Input
→ State Update
→ Output Projection

This layer can replace attention blocks in sequence models.



# 8. HiPPO

HiPPO = High-order Polynomial Projection Operator

Purpose:

Store maximum information about history inside a compact state.

Benefits:

- Long-term memory
- Better sequence compression
- Foundation for S4



# 9. S4 Architecture

S4 combines:

1. State Space Models
2. HiPPO initialization
3. Structured matrices

Benefits:

- Linear scaling
- Long-context modeling
- Competitive with transformers on long sequences



# 10. Limitations of SSM and S4

Problems:

- Fixed dynamics
- Weak content awareness
- Limited selective retrieval
- Difficulty matching transformer induction behavior

These limitations motivated Mamba.



# 11. Selective Copying

Example:

A B C D X

Need to remember X and forget others.

Traditional SSM:
Treats tokens similarly.

Desired:
Content-aware memory updates.



# 12. Induction Heads

Transformer induction heads learn patterns such as:

A B C ... A B ?

Predict:

C

This mechanism contributes strongly to in-context learning.



# 13. Mamba

Core innovation:

Make state-space parameters input dependent.

Instead of:

A, B, C

Use:

A(x), B(x), C(x)

Now the model decides:

- what to remember
- what to forget
- what to retrieve



# 14. Selective Scan

Problem:

Input-dependent parameters break convolution efficiency.

Solution:

Selective Scan

Benefits:

- Linear complexity
- Dynamic memory
- Efficient parallel execution



# 15. Kernel Fusion & Hardware Awareness

Mamba optimizes memory movement rather than only FLOPs.

Techniques:

- Kernel Fusion
- GPU cache optimization
- Reduced HBM traffic

Result:

Extremely fast inference for long sequences.



# 16. Future

Hybrid architectures combine attention and Mamba.

Examples:

- Jamba
- Zamba

Future systems will likely mix:

- Attention for reasoning
- Mamba for memory
- MoE for scale



# Hands-On 1: Long Text Summarization Using a Pretrained Transformer

Although Mamba checkpoints may not always be available in every environment,
we can compare long-sequence behavior using pretrained transformer models.


In [2]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

text = '''
State Space Models are becoming an alternative to transformers
for efficient long-context processing. Mamba introduces selective
state spaces and selective scan algorithms that enable linear-time
sequence modeling.
'''

result = summarizer(text, max_length=50, min_length=20)
print(result[0]["summary_text"])


Device set to use cpu
Your max_length is set to 50, but your input_length is only 43. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=21)


State Space Models are becoming an alternative to transformers for efficient long-context processing. Mamba introduces selective state spaces and selective scan algorithms that enable linear-timesequence modeling.



# Hands-On 2: Generate Text with GPT-2

Observe autoregressive generation.


In [3]:

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

output = generator(
    "State Space Models are important because",
    max_new_tokens=40
)

print(output[0]["generated_text"])


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


State Space Models are important because they give you information about your data, and also help you to make informed decisions about how you use these models.

The following table shows how you can download data for the NASA Space Launch System



# Hands-On 3: Compare Complexity

Attention:

O(N²)

Mamba:

O(N)

Simple numerical comparison below.


In [4]:

for N in [1000, 5000, 10000]:
    attention = N**2
    mamba = N

    print(f"Sequence={N}")
    print(f"Attention complexity ≈ {attention:,}")
    print(f"Mamba complexity     ≈ {mamba:,}")
    print("-"*40)


Sequence=1000
Attention complexity ≈ 1,000,000
Mamba complexity     ≈ 1,000
----------------------------------------
Sequence=5000
Attention complexity ≈ 25,000,000
Mamba complexity     ≈ 5,000
----------------------------------------
Sequence=10000
Attention complexity ≈ 100,000,000
Mamba complexity     ≈ 10,000
----------------------------------------


# Load a pretrained Mamba model

In [5]:
from transformers import AutoTokenizer, MambaForCausalLM

model_name = "state-spaces/mamba-130m-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MambaForCausalLM.from_pretrained(model_name)

prompt = "State Space Models are"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

C:\ProgramData\anaconda3\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\atanu\.cache\huggingface\hub\models--state-spaces--mamba-130m-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn,

State Space Models are used to model the dynamics of the system. The model is a two-dimensional system of coupled differential equations. The model is solved using the finite difference method. The model is solved using the finite difference method with a time step of 0.01.



# Interview Questions

1. Why do transformers scale quadratically?
2. What is a state vector?
3. Explain A, B, C matrices.
4. What is discretization?
5. What is HiPPO?
6. Why was S4 important?
7. What problem does Mamba solve?
8. What is selective copying?
9. What is selective scan?
10. Why is Mamba hardware-aware?
11. Difference between transformer and Mamba?
12. What are Jamba and Zamba?
